# Retail Customer Purchase Intelligence — Milestone 2
**Participant:** Participant_M2_02  
**Project:** Retail Customer Purchase Intelligence  
**Milestone:** 2  
**Dataset package:** `Retail.zip`

## Objective
Predict purchase amounts, segment customers, understand purchase drivers, and support marketing/inventory decisions.

## Milestone scope
Decision Tree, Random Forest, and boosting regression; tuning; compare MAE, RMSE, R²; feature importance; cluster customers using aggregated purchase behaviour.

> This is a synthetic sample submission for evaluator testing. Dataset files must be placed in the paths referenced below. Results are not pre-filled or fabricated.

## Model comparison: purchase amount regression
Use identical train/test partitions and compare tree-based regressors with a transparent baseline. The notebook does not assume a winner before measuring it.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

demo = pd.read_csv(Path("data") / "user_demographics.csv")
purchase = pd.read_csv(Path("data") / "User_product_purchase_details.csv")
df = purchase.merge(demo, on="User_ID", how="left", validate="many_to_one")
y = pd.to_numeric(df["Purchase"], errors="coerce")
X = df.drop(columns=["Purchase", "User_ID", "Product_ID"], errors="ignore")
ok = y.notna(); X, y = X.loc[ok], y.loc[ok]
cat = X.select_dtypes(include=["object", "category"]).columns.tolist()
num = X.select_dtypes(include=np.number).columns.tolist()
prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat)
])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)
models = {
    "DecisionTree": DecisionTreeRegressor(max_depth=8, random_state=42),
    "RandomForest": RandomForestRegressor(n_estimators=150, max_depth=14, random_state=42, n_jobs=-1),
    "HistGradientBoosting": HistGradientBoostingRegressor(max_iter=150, random_state=42)
}
rows = []
for name, estimator in models.items():
    pipe = Pipeline([("prep", prep), ("model", estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    rows.append({"model": name, "MAE": mean_absolute_error(y_test, pred),
                 "RMSE": mean_squared_error(y_test, pred)**.5, "R2": r2_score(y_test, pred)})
results = pd.DataFrame(rows).sort_values("RMSE")
display(results)

## Customer segmentation
Aggregate purchase behaviour per customer before clustering. Select and scale numeric features deliberately; evaluate cluster quality rather than assigning business meaning from cluster numbers alone.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

customer = df.groupby("User_ID", as_index=False).agg(
    total_purchase=("Purchase", "sum"),
    mean_purchase=("Purchase", "mean"),
    transaction_count=("Purchase", "count")
)
Z = StandardScaler().fit_transform(customer[["total_purchase", "mean_purchase", "transaction_count"]])
km = KMeans(n_clusters=3, random_state=42, n_init=10)
customer["cluster"] = km.fit_predict(Z)
print("silhouette:", silhouette_score(Z, customer["cluster"]))
display(customer.groupby("cluster")[["total_purchase", "mean_purchase", "transaction_count"]].mean())